# Geração de Prontuários Sintéticos no Google Colab

Notebook para executar a geração de prontuários sintéticos usando o código do repositório e o Google Drive.
O fluxo usa `MedQuAD` e `PubMedQA` como fonte, mantém checkpoint de sucesso e falha, e pode salvar a saída no Drive.
Também pode rodar com backend local `llama` usando GPU do Colab.

Fluxo:
1. Montar o Google Drive
2. Clonar ou atualizar o repositório por HTTPS
3. Instalar dependências
4. Configurar a execução da T3
5. Rodar o CLI de geração de prontuários sintéticos


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!if [ -d /content/tech-challenge-fase-3/.git ]; then git -C /content/tech-challenge-fase-3 pull; else git clone https://github.com/JeffersonPantoja/tech-challenge-fase3.git /content/tech-challenge-fase-3; fi


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/tech-challenge-fase-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/teach-chalenge3')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert PROJECT_ROOT.exists(), f'Projeto não encontrado em {PROJECT_ROOT}'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Projeto:', PROJECT_ROOT)
print('Drive:', DRIVE_ROOT)


In [ ]:
!pip -q install --upgrade openai python-dotenv transformers accelerate peft bitsandbytes sentencepiece torch


In [ ]:
BACKEND = 'llama'  # ou 'openai'
OPENAI_MODEL = 'gpt-4o-mini'
LLAMA_MODEL_PATH = '/content/drive/MyDrive/teach-chalenge3/llama-model'
LLAMA_MAX_NEW_TOKENS = 512
SAVE_TO_DRIVE = True
RESUME = True
BATCH_SIZE = 100
NUM_BATCHES = None
OUTPUT_NAME = 'patient_records.jsonl'
CHECKPOINT_NAME = 'patient_records.checkpoint.json'
FAILED_CHECKPOINT_NAME = 'patient_records.failed.checkpoint.json'

print('Backend:', BACKEND)
print('Modelo OpenAI:', OPENAI_MODEL)
print('Modelo Llama:', LLAMA_MODEL_PATH)
print('Salvar no Drive:', SAVE_TO_DRIVE)


In [ ]:
args = [
    'python3', '-m', 'src.main_synthetic_records',
    '--resources-dir', str(PROJECT_ROOT / 'resources'),
    '--output', OUTPUT_NAME,
    '--checkpoint', CHECKPOINT_NAME,
    '--failed-checkpoint', FAILED_CHECKPOINT_NAME,
    '--batch-size', str(BATCH_SIZE),
    '--backend', BACKEND,
    '--openai-model', OPENAI_MODEL,
    '--llama-model-path', LLAMA_MODEL_PATH,
    '--llama-max-new-tokens', str(LLAMA_MAX_NEW_TOKENS),
]

if SAVE_TO_DRIVE:
    args.extend(['--save-to-drive', '--drive-dir', str(DRIVE_ROOT)])

if RESUME:
    args.append('--resume')
else:
    args.append('--no-resume')

if NUM_BATCHES is not None:
    args.extend(['--num-batches', str(NUM_BATCHES)])

print('Comando:', ' '.join(args))
!{' '.join(args)}
